# 3. Feature Selection

**Homework — Group 34**

This notebook starts from the preprocessing notebook's hand-off (sections cited as 2.x) and decides which columns the
models use.

**Strategy.** Three kinds of method vote on each column: a **filter** (relevance and redundancy), an **embedded** method
(L1-penalised logistic regression) and a **wrapper** (recursive feature elimination). Each method runs on the training
part of each of the five time folds, so the validation rows never influence the choice: 3 methods × 5 folds = 15 votes
per column. A column is kept if it gets a majority (at least 8 of 15) and more votes than the probe below. The selected set is then compared with all columns
on the same folds; if the scores are inside the noise, the smaller set is kept, since it is simpler and less likely to
overfit.

`random_noise` is added back as a **probe**: a column known to carry no information. A column that does no better than
it should not be trusted.

### 3.0 Setup and hand-off

In [ ]:
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
OUTPUT_FILE = DATA_DIR / "selected_features.joblib"   # read by the modelling notebook

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # IterativeImputer early-stopping notice
pd.set_option("display.width", 200)

In [ ]:
handoff = joblib.load(DATA_DIR / "preprocessed.joblib")
X_train, y, X_test = handoff["X_train"], handoff["y"], handoff["X_test"]
dates_train, cross_v, INSTR_C = handoff["dates_train"], handoff["cross_v"], handoff["instrument_C"]
NUM, FLAGS, CAT = (handoff["column_groups"][k] for k in ("num", "flags", "cat"))
CANDIDATES = NUM + FLAGS + CAT

# the probe: random_noise is read again from train.csv and matched by ID
PROBE = "random_noise"
X_train = X_train.assign(**{PROBE: pd.read_csv(DATA_DIR / "train.csv", index_col="ID")[PROBE].loc[X_train.index]})

print(f"candidates: {len(NUM)} numeric, {len(FLAGS)} flags, {len(CAT)} categorical = {len(CANDIDATES)} columns (+ probe)")
print(f"rows: {len(X_train)} | folds: {len(cross_v)} | C of the measuring model: {INSTR_C}")

### 3.1 Tools

- `preprocessor_for(cols)` reuses the preprocessing recipe from the hand-off, restricted to `cols`. The probe is passed
  through as it is: it is already centred and scaled, and its negative values cannot be logged;
- `source_column` maps each pipeline output back to its original column, so a categorical column counts as selected if
  any of its one-hot columns is;
- `cv_auc` and `paired_table` are the scoring and comparison tools of 2.7, with columns instead of a recipe.

In [ ]:
def preprocessor_for(cols):
    """The preprocessing recipe of the hand-off, restricted to `cols` (plus the probe, passed through, if listed)."""
    prep = clone(handoff["preprocessor"])
    blocks = [(name, step, [c for c in block if c in cols]) for name, step, block in prep.transformers]
    if PROBE in cols:
        blocks.append(("probe", "passthrough", [PROBE]))
    prep.set_params(transformers=[b for b in blocks if b[2]])
    return prep.set_output(transform="pandas")


def source_column(feature):
    """Original column behind a pipeline output, e.g. 'cat__payment_type_Payment 9' -> 'payment_type'."""
    name = feature.split("__", 1)[1]
    return next(c for c in CAT if name.startswith(c + "_")) if feature.startswith("cat__") else name


def time_splits_with_horizon(dates, n_splits=5, horizon_days=90):
    """Time folds with the 90-day gap of 2.6 (used here inside a training fold, for RFECV)."""
    for tr_idx, va_idx in TimeSeriesSplit(n_splits=n_splits).split(dates):
        keep = dates.iloc[tr_idx] < dates.iloc[va_idx[0]] - pd.Timedelta(days=horizon_days)
        yield tr_idx[keep.to_numpy()], va_idx


def cv_auc(cols, clf=None):
    """ROC-AUC on each of the five folds, preprocessing and model fitted on the training part only."""
    scores = []
    for tr, va in cross_v:
        model = Pipeline([("prep", preprocessor_for(cols)),
                          ("clf", clone(clf) if clf is not None else LogisticRegression(C=INSTR_C, max_iter=5000))])
        model.fit(X_train.iloc[tr], y.iloc[tr])
        scores.append(roc_auc_score(y.iloc[va], model.predict_proba(X_train.iloc[va])[:, 1]))
    return np.array(scores)


def paired_table(runs, reference):
    """Fold-by-fold comparison with `reference`: better / worse only beyond 2 standard errors and on all five folds."""
    rows = []
    for name, s in runs.items():
        d = s - runs[reference]
        sem = d.std(ddof=1) / np.sqrt(len(d))
        wins, losses = (d > 0).sum(), (d < 0).sum()
        if name == reference:
            verdict = "reference"
        elif d.mean() > 2 * sem and wins == len(d):
            verdict = "better"
        elif -d.mean() > 2 * sem and losses == len(d):
            verdict = "worse"
        else:
            verdict = "inside the noise"
        rows.append({"version": name, "AUC": s.mean(), "dAUC": d.mean(), "SEM": sem,
                     "wins": f"{wins}/{len(d)}", "verdict": verdict})
    table = pd.DataFrame(rows).set_index("version")
    print(table.to_string(formatters={"AUC": "{:.4f}".format, "dAUC": "{:+.4f}".format, "SEM": "{:.4f}".format}))
    return table


# the methods are shown on the largest training fold; section 3.5 runs them on all five
tr_last = cross_v[-1][0]
X_last, y_last, dates_last = X_train.iloc[tr_last], y.iloc[tr_last], dates_train.iloc[tr_last]

### 3.2 Filter: relevance and redundancy

- **Relevance.** A numeric column or flag must be more correlated with the target (absolute Spearman) than the probe.
  A categorical column must pass a chi-square test of independence with the target (p < 0.05);
- **Redundancy.** EDA 1.13 found groups of near-copies. Going from the most to the least relevant column, a column is
  skipped if its Spearman correlation with a column already kept exceeds 0.9, so one column per group survives: the one
  most related to the target.

The groups on all of train, for reference:

In [ ]:
rho_target = X_train[NUM + FLAGS].corrwith(y, method="spearman").abs()
rho = X_train[NUM + FLAGS].corr(method="spearman").abs()
pairs = [(a, b, rho.loc[a, b]) for i, a in enumerate(rho.index) for b in rho.index[i + 1:] if rho.loc[a, b] > 0.9]
print(pd.DataFrame([{"column A": a, "column B": b, "rho": round(r, 3),
                     "|rho| A-target": round(rho_target[a], 3), "|rho| B-target": round(rho_target[b], 3)}
                    for a, b, r in sorted(pairs, key=lambda p: -p[2])]).to_string(index=False))

In [ ]:
REDUNDANT_RHO = 0.9


def filter_keep(X, y):
    rho_y = X[NUM + FLAGS + [PROBE]].corrwith(y, method="spearman").abs()
    relevant = [c for c in NUM + FLAGS if rho_y[c] > rho_y[PROBE]]

    rho_x = X[relevant].corr(method="spearman").abs().fillna(0)   # undefined (no overlapping values) = not redundant
    kept = []
    for c in sorted(relevant, key=rho_y.get, reverse=True):
        if all(rho_x.loc[c, k] <= REDUNDANT_RHO for k in kept):
            kept.append(c)

    for c in CAT:
        p_value = chi2_contingency(pd.crosstab(X[c].fillna("missing"), y))[1]
        if p_value < 0.05:
            kept.append(c)
    return kept


kept = filter_keep(X_last, y_last)
print(f"filter keeps {len(kept)} of {len(CANDIDATES)} columns on the largest training fold")
print("dropped:", sorted(set(CANDIDATES) - set(kept)))

<div class="alert alert-block alert-warning">

- The redundant groups match EDA 1.13: the three order totals (rho >= 0.99), each flag with the count it is built from (`bought_30d` with `purchase_count_30d`, 0.98), count and spend in the same window, and three pairs of history columns;
- On the largest training fold every column beats the probe on relevance, so the filter's drops all come from redundancy: two of the three order totals, `bought_30d` and `bought_90d`, the 30- and 90-day spends, `recent_share_30d`, and the weaker member of each history pair (`purchases_per_year`, `prior_total_spend`, `prior_std_purchase_value`);
- All four categorical columns pass the chi-square test.

</div>

### 3.3 Embedded: L1-penalised logistic regression

An L1 penalty sets the coefficients of unhelpful columns to exactly zero, so the columns left with a non-zero coefficient
are the ones the model chose. Its strength `C` is chosen first, by fold ROC-AUC with every candidate and the probe in the
model. A smaller `C` removes more columns, so the **one-standard-error rule** is used: the smallest `C` whose mean AUC
is within one standard error of the best one.

In [ ]:
def lasso(C):
    return LogisticRegression(l1_ratio=1.0, solver="liblinear", C=C, max_iter=5000)   # l1_ratio=1: pure L1 penalty


LASSO_GRID = [0.01, 0.03, 0.1, 0.3, 1.0]
runs_lasso = {f"C={C}": cv_auc(CANDIDATES + [PROBE], lasso(C)) for C in LASSO_GRID}
best = max(runs_lasso, key=lambda name: runs_lasso[name].mean())
table = paired_table(runs_lasso, best)
within_1se = table.index[-table["dAUC"] <= table["SEM"]]
LASSO_C = min(float(name.removeprefix("C=")) for name in within_1se)
print(f"\nC of the L1 model: {LASSO_C} (smallest C within one standard error of {best})")

In [ ]:
def lasso_keep(X, y):
    Z = preprocessor_for(CANDIDATES + [PROBE]).fit_transform(X)
    coef = pd.Series(lasso(LASSO_C).fit(Z, y).coef_[0], index=Z.columns)
    return sorted({source_column(f) for f in coef.index[coef != 0]})


kept = lasso_keep(X_last, y_last)
print(f"L1 keeps {len(kept)} of {len(CANDIDATES) + 1} columns (probe included) on the largest training fold")
print("dropped:", sorted(set(CANDIDATES + [PROBE]) - set(kept)))

<div class="alert alert-block alert-warning">

- C = 0.1 and 0.3 score the same; by the one-standard-error rule the sparser C = 0.1 is used. Smaller values lose clearly (C = 0.01 costs 0.016 AUC);
- On the largest training fold the L1 model keeps 23 of the 40 columns. Like the filter, it drops the duplicate order totals and the flags, and keeps one column out of several correlated history columns;
- It also keeps the probe: an L1 penalty this weak does not reject pure noise on its own, which is why no single method decides.

</div>

### 3.4 Wrapper: recursive feature elimination

RFE fits the logistic regression, removes the output column with the smallest coefficient, and repeats. `RFECV` picks
how many columns to keep by ROC-AUC on three inner time folds (with the same 90-day gap) inside the training fold, so the
number is not chosen by hand.

In [ ]:
def rfe_keep(X, y, dates):
    Z = preprocessor_for(CANDIDATES + [PROBE]).fit_transform(X)
    inner = list(time_splits_with_horizon(dates, n_splits=3))
    rfe = RFECV(LogisticRegression(C=INSTR_C, max_iter=5000), step=1, cv=inner, scoring="roc_auc")
    rfe.fit(Z, y)
    return sorted({source_column(f) for f in Z.columns[rfe.support_]})


kept = rfe_keep(X_last, y_last, dates_last)
print(f"RFE keeps {len(kept)} of {len(CANDIDATES) + 1} columns (probe included) on the largest training fold")
print("dropped:", sorted(set(CANDIDATES + [PROBE]) - set(kept)))

<div class="alert alert-block alert-warning">

- RFECV keeps 28 of the 40 columns on the largest training fold, the number that scored best on the inner folds;
- It agrees with the other two methods on the flags and on two of the three order totals, but keeps the probe as well: a wrapper can pick up a noise column that happens to help on a small inner fold.

</div>

### 3.5 Votes on all five folds

Each method runs on the training part of each fold. The table counts, per column, how many times each method kept it.
A column is kept if it has a majority (at least 8 of 15) **and** more votes than the probe: a column chosen no more
often than pure noise has not shown it carries information.

In [ ]:
votes = pd.DataFrame(0, index=CANDIDATES + [PROBE], columns=["filter", "lasso", "rfe"])
for tr, va in cross_v:
    X_fold, y_fold = X_train.iloc[tr], y.iloc[tr]
    votes.loc[filter_keep(X_fold, y_fold), "filter"] += 1
    votes.loc[lasso_keep(X_fold, y_fold), "lasso"] += 1
    votes.loc[rfe_keep(X_fold, y_fold, dates_train.iloc[tr]), "rfe"] += 1
votes["total"] = votes.sum(axis=1)

MIN_VOTES = max(8, votes.loc[PROBE, "total"] + 1)   # a majority of 15, and more than the probe
votes["kept"] = (votes["total"] >= MIN_VOTES) & (votes.index != PROBE)
print(f"probe votes: {votes.loc[PROBE, 'total']} -> a column needs at least {MIN_VOTES} of 15\n")
print(votes.sort_values(["total", "filter"], ascending=False).to_string())

<div class="alert alert-block alert-warning">

- **The probe gets 7 of 15 votes** (the L1 model and RFE kept it in some folds), so a column needs at least 8. Every column at or below the probe is dropped, whatever a single method said;
- **27 of the 39 candidates are kept.** Three are kept by every method in every fold: `N_ Detalhes`, `prior_mean_purchase_value` and `purchase_count_90d`. All four categorical columns are kept;
- **Dropped as redundant:** `Total a Pagar` and `Total Liquido` (near-copies of `Total Bruto`), `bought_30d` and `bought_90d` (the counts they come from are kept), `prior_total_spend`, `prior_std_purchase_value` and the 90- and 365-day spends;
- **Dropped for no more votes than the probe:** `Total Bruto`, `prior_min_purchase_value`, `has_purchased_before` and `days_since_previous_purchase`. The filter keeps them in every fold, but the two model-based methods rarely do: order totals barely relate to the target (EDA 1.13), `prior_purchase_count` already tells the model when the history is empty, and the purchase counts over 30 and 90 days, which are kept, already describe how recent the last purchases were;
- `customer_id` was not a candidate: it is a label, excluded in 2.9.

</div>

### 3.6 Final feature set

The selected columns are compared with all candidates on the five folds, with the measuring model of 2.8.

In [ ]:
SELECTED = [c for c in CANDIDATES if votes.loc[c, "kept"]]
print(f"selected: {len(SELECTED)} of {len(CANDIDATES)} columns")
for name, group in [("numeric", NUM), ("flags", FLAGS), ("categorical", CAT)]:
    print(f"  {name:12s} {[c for c in group if c in SELECTED]}")
print()
_ = paired_table({"all candidates": cv_auc(CANDIDATES), "selected": cv_auc(SELECTED)}, "all candidates")

<div class="alert alert-block alert-warning">

- The 27 selected columns score the same as all 39 (+0.0005, inside the noise). By the rule stated at the top, the smaller set is kept: it is simpler, and with fewer redundant columns the models depend less on any one of several near-copies;
- The selection was made with linear methods; non-linear models in the modelling notebook might use a dropped column differently. The vote table is saved with the hand-off, so a dropped column can be tested again there.

</div>

### 3.7 Hand-off

The modelling notebook receives the selected columns and the preprocessing recipe restricted to them, **unfitted**, so
each fold refits it.

In [ ]:
selection = {
    "X_train": X_train[SELECTED],                   # selected columns only (probe removed)
    "y": y,
    "X_test": X_test[SELECTED],                     # same columns, in sample_submission order
    "dates_train": dates_train,
    "cross_v": cross_v,
    "preprocessor": preprocessor_for(SELECTED),     # the recipe of 2.x on the selected columns, unfitted
    "selected": SELECTED,
    "votes": votes,
    "instrument_C": INSTR_C,
}
joblib.dump(selection, OUTPUT_FILE)
print("saved to", OUTPUT_FILE, "|", selection["X_train"].shape, selection["X_test"].shape)